# Red neuronal ANN(2, b, c) a mano

Una red con 2 entradas, `b` neuronas ocultas y `c` salidas, todas con activación sigmoide.
Mueve los pesos y sesgos y mira cómo cambian las ecuaciones, la superficie de cada salida
y el grafo de la red. Aquí no se entrena nada: los pesos los pones tú.

Librerías: `conda install -n profesia numpy matplotlib networkx ipywidgets`

In [1]:
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
from IPython.display import Math, clear_output, display
from ipywidgets import HTML, FloatSlider, HBox, IntSlider, Layout, Output, VBox

# ==========================================
# 1. DEFINICIÓN DE CONTROLES Y WIDGETS
# ==========================================

# Hiperparámetro 'a' es fijo (a = 2)
# Hiperparámetro 'c' está limitado entre 1 y 3
slider_b = IntSlider(
    value=2,
    min=1,
    max=4,
    description="Ocultas (b):",
    style={"description_width": "initial"},
)
slider_c = IntSlider(
    value=1,
    min=1,
    max=3,
    description="Salidas (c):",
    style={"description_width": "initial"},
)

hyper_box = VBox(
    [
        HTML(
            "<h3 style='margin-bottom:5px; color:#2c3e50;'>Arquitectura ANN(2, b, c)</h3>"
        ),
        HTML(
            "<p style='margin: 0 0 10px 0; color: #7f8c8d;'><i>* Entradas (a) fijadas en 2 ($x_1, x_2$)</i></p>"
        ),
        slider_b,
        slider_c,
    ],
    layout=Layout(
        flex="1",
        padding="10px",
        border="1px solid #bdc3c7",
        border_radius="8px",
        background_color="#fdfdfd",
    ),
)

# Panel de la Ecuación Matemática
latex_box = Output(
    layout=Layout(
        flex="2",
        padding="10px",
        border="1px solid #bdc3c7",
        border_radius="8px",
        margin="0 0 0 20px",
        background_color="#ffffff",
        overflow="auto",
    )
)

out_plot = Output()

# Ecualizadores para Pesos (W1) y Sesgos (b1) - Capa Oculta
W1_sliders = [
    [
        FloatSlider(
            value=np.round(np.random.randn(), 1),
            min=-5,
            max=5,
            step=0.1,
            description=f"Peso w_{j + 1}{i + 1}:",
            layout=Layout(width="180px"),
        )
        for i in range(2)
    ]
    for j in range(4)
]
b1_sliders = [
    FloatSlider(
        value=0,
        min=-5,
        max=5,
        step=0.1,
        description=f"Sesgo b1_{j + 1}:",
        layout=Layout(width="180px"),
    )
    for j in range(4)
]

# Ecualizadores para Pesos (W2) y Sesgos (b2) - Capa de Salida
W2_sliders = [
    [
        FloatSlider(
            value=np.round(np.random.randn(), 1),
            min=-5,
            max=5,
            step=0.1,
            description=f"Peso v_{k + 1}{j + 1}:",
            layout=Layout(width="180px"),
        )
        for j in range(4)
    ]
    for k in range(3)
]
b2_sliders = [
    FloatSlider(
        value=0,
        min=-5,
        max=5,
        step=0.1,
        description=f"Sesgo b2_{k + 1}:",
        layout=Layout(width="180px"),
    )
    for k in range(3)
]

# ==========================================
# 2. FUNCIONES MATEMÁTICAS Y DE RENDERIZADO
# ==========================================


def sigmoid(x):
    """Función sigmoide, 1 / (1 + e^-x)."""
    return 1 / (1 + np.exp(-np.clip(x, -500, 500)))


def update_latex(b, c):
    """Construye y renderiza las ecuaciones expandidas para todas las salidas 'y'"""
    with latex_box:
        clear_output(wait=True)
        display(
            HTML(
                "<h3 style='margin-top:0; color:#2c3e50;'>Ecuaciones Expandidas (Valores en Tiempo Real)</h3>"
            )
        )

        for k in range(c):
            hidden_terms = []
            for j in range(b):
                # w11*x1 + w12*x2 + b1
                inner_terms = [
                    f"{W1_sliders[j][i].value:.1f} x_{i + 1}" for i in range(2)
                ]
                inner_expr = " + ".join(inner_terms) + f" + {b1_sliders[j].value:.1f}"
                inner_expr = inner_expr.replace("+ -", "- ")

                v_val = W2_sliders[k][j].value
                hidden_terms.append(f"{v_val:.1f} \sigma({inner_expr})")

            outer_expr = " + ".join(hidden_terms) + f" + {b2_sliders[k].value:.1f}"
            outer_expr = outer_expr.replace("+ -", "- ")

            formula = rf"y_{k + 1} = \sigma \Big( {outer_expr} \Big)"
            display(Math(formula))

        display(
            HTML(
                "<p style='font-size:0.85em; color:#7f8c8d; margin-top: 15px;'>* Las superficies 3D inferiores grafican cada una de estas funciones resultantes.</p>"
            )
        )


def update_all(*args):
    """Actualiza las ecuaciones y renderiza dinámicamente múltiples gráficos 3D"""
    a = 2  # Fijo
    b = slider_b.value
    c = slider_c.value

    update_latex(b, c)

    with out_plot:
        clear_output(wait=True)

        # Ajuste dinámico de la cuadrícula de gráficos dependiendo de c
        if c == 1:
            fig = plt.figure(figsize=(14, 6))
            rows, cols, net_pos = 1, 2, 2
        elif c == 2:
            fig = plt.figure(figsize=(18, 6))
            rows, cols, net_pos = 1, 3, 3
        else:  # c == 3
            fig = plt.figure(figsize=(16, 12))
            rows, cols, net_pos = 2, 2, 4

        # Generar datos del plano XY
        x_vals = np.linspace(-5, 5, 30)
        y_vals = np.linspace(-5, 5, 30)
        X1, X2 = np.meshgrid(x_vals, y_vals)

        # Calcular Capa Oculta H
        H = []
        for j in range(b):
            sum_h = (
                W1_sliders[j][0].value * X1
                + W1_sliders[j][1].value * X2
                + b1_sliders[j].value
            )
            H.append(sigmoid(sum_h))

        # Calcular e imprimir Capa de Salida Z para cada neurona c
        for k in range(c):
            Z = b2_sliders[k].value
            for j in range(b):
                Z += W2_sliders[k][j].value * H[j]
            Z = sigmoid(Z)

            # Crear subgráfico 3D para y_k
            ax = fig.add_subplot(rows, cols, k + 1, projection="3d")
            surf = ax.plot_surface(
                X1, X2, Z, cmap="coolwarm", alpha=0.9, edgecolor="none"
            )
            ax.set_title(f"Superficie de Decisión ($y_{k + 1}$)", fontsize=14)
            ax.set_xlabel("Entrada $x_1$")
            ax.set_ylabel("Entrada $x_2$")
            ax.set_zlabel(f"Salida $y_{k + 1}$ (0 a 1)")
            ax.set_zlim(0, 1)

        # --- GRAFO DE LA RED NEURONAL ---
        ax_net = fig.add_subplot(rows, cols, net_pos)
        ax_net.set_title(f"Arquitectura ANN({a}, {b}, {c})", fontsize=14)
        G = nx.DiGraph()

        pos = {}
        for i in range(a):
            pos[f"x{i + 1}"] = (0, (a - 1) / 2 - i)
        for j in range(b):
            pos[f"h{j + 1}"] = (1, (b - 1) / 2 - j)
        for k in range(c):
            pos[f"y{k + 1}"] = (2, (c - 1) / 2 - k)

        G.add_nodes_from(pos.keys())

        # Enlaces
        for j in range(b):
            for i in range(a):
                G.add_edge(f"x{i + 1}", f"h{j + 1}", weight=W1_sliders[j][i].value)
        for k in range(c):
            for j in range(b):
                G.add_edge(f"h{j + 1}", f"y{k + 1}", weight=W2_sliders[k][j].value)

        # Estilos del Grafo
        node_colors = ["#3498db"] * a + ["#2ecc71"] * b + ["#e74c3c"] * c
        nx.draw_networkx_nodes(
            G,
            pos,
            ax=ax_net,
            node_color=node_colors,
            node_size=1800,
            edgecolors="white",
            linewidths=2,
        )

        labels = {f"x{i + 1}": f"x{i + 1}" for i in range(a)}
        labels.update(
            {f"h{j + 1}": f"h{j + 1}\nb={b1_sliders[j].value:.1f}" for j in range(b)}
        )
        labels.update(
            {f"y{k + 1}": f"y{k + 1}\nb={b2_sliders[k].value:.1f}" for k in range(c)}
        )
        nx.draw_networkx_labels(
            G,
            pos,
            labels=labels,
            font_size=9,
            font_color="white",
            font_weight="bold",
            ax=ax_net,
        )

        nx.draw_networkx_edges(
            G, pos, ax=ax_net, edge_color="#7f8c8d", arrows=True, arrowsize=15
        )
        edge_labels = {(u, v): f"{d['weight']:.1f}" for u, v, d in G.edges(data=True)}
        nx.draw_networkx_edge_labels(
            G, pos, edge_labels=edge_labels, font_size=8, ax=ax_net, label_pos=0.7
        )

        plt.axis("off")
        plt.tight_layout()
        plt.show()


def update_ui(*args):
    """Muestra u oculta los sliders según los hiperparámetros b y c"""
    a = 2
    b = slider_b.value
    c = slider_c.value

    # Capa 1
    for j in range(4):
        for i in range(2):
            W1_sliders[j][i].layout.display = "flex" if (j < b and i < a) else "none"
        b1_sliders[j].layout.display = "flex" if j < b else "none"

    # Capa 2
    for k in range(3):
        for j in range(4):
            W2_sliders[k][j].layout.display = "flex" if (k < c and j < b) else "none"
        b2_sliders[k].layout.display = "flex" if k < c else "none"

    update_all()


# ==========================================
# 3. EVENTOS Y EMPAQUETADO VISUAL
# ==========================================

slider_b.observe(update_ui, "value")
slider_c.observe(update_ui, "value")

for group in [W1_sliders, W2_sliders]:
    for row in group:
        for sl in row:
            sl.continuous_update = (
                False  # Actualizar solo al soltar el clic para evitar lag visual
            )
            sl.observe(update_all, "value")

for group in [b1_sliders, b2_sliders]:
    for sl in group:
        sl.continuous_update = False
        sl.observe(update_all, "value")

w1_box = VBox(
    [
        HTML(
            "<h4 style='color:#2980b9; margin-bottom:5px;'>Pesos Capa Oculta (w) y Sesgos (b1)</h4>"
        )
    ]
    + [HBox(W1_sliders[j] + [b1_sliders[j]]) for j in range(4)]
)
w2_box = VBox(
    [
        HTML(
            "<h4 style='color:#e74c3c; margin-bottom:5px;'>Pesos Capa Salida (v) y Sesgos (b2)</h4>"
        )
    ]
    + [HBox(W2_sliders[k] + [b2_sliders[k]]) for k in range(3)]
)

sliders_panel = HBox([w1_box, w2_box], layout=Layout(margin="20px 0"))
top_panel = HBox([hyper_box, latex_box], layout=Layout(align_items="stretch"))

# Ejecutar el estado inicial por defecto
update_ui()
display(top_panel, sliders_panel, out_plot)

<>:148: SyntaxWarning: invalid escape sequence '\s'
<>:148: SyntaxWarning: invalid escape sequence '\s'
/tmp/ipykernel_2148991/4166535100.py:148: SyntaxWarning: invalid escape sequence '\s'
  hidden_terms.append(f"{v_val:.1f} \sigma({inner_expr})")


Output()